# ⚙️ ISE498 Workshop: Engine Production Analysis with Gemini in Colab

**Part 1 of 2: analyze the data with Gemini.** In Part 2 you turn your results into a formatted report with Claude.

## Data Science Support
- **GetDataHelp**: [go.ncsu.edu/getdatahelp](https://go.ncsu.edu/getdatahelp)
- **Email**: [getdatahelp@ncsu.edu](mailto:getdatahelp@ncsu.edu)

## Session at a Glance
- **Length**: 60 minutes (about 35 in this notebook, 15 with Claude, 10 for the intro and wrap-up)
- **Level**: Beginner to intermediate (you can read basic pandas code)
- **You need**: a Google account (18+) that can use Gemini in Colab, and a Claude account at [claude.ai](https://claude.ai) for Part 2

> **First:** choose **File → Save a copy in Drive**. Changes to a notebook opened from GitHub aren't saved
> automatically, and your own copy also skips the "not authored by Google" warning.

## Learning Objectives
By the end of this workshop, you will be able to:
1. Direct Gemini in Colab with precise, decision-focused prompts
2. Verify AI-generated code and results against checkpoints before trusting them
3. Find where quality and downtime losses concentrate in production data
4. Package results as an evidence pack that a report writer can trust
5. Use Claude to turn the evidence pack into a formatted report, then fact-check it

## How This Workshop Works

```text
 Colab + Gemini          Evidence pack               Claude                       You
 analyze the data  ───▶  findings.md + charts  ───▶  writes the report    ───▶    fact-check it
 (Steps 0-5)             (Step 6, built by code)     (Part 2: chat, optional agent)
```

**The rule that makes this work:** the report may only use numbers that code computed. Gemini helps you
write and run the analysis, one cell packages the results, and Claude writes from that package, not from
memory or guesswork.

## Working with Gemini in Colab

You'll use three Gemini features (labels can shift slightly as Colab updates). The first time, Colab may ask
you to accept its AI terms.

| Feature | Where to find it | Use it for |
|---|---|---|
| **Gemini chat panel** | The **Gemini** spark icon (✦) at the bottom of the notebook | Paste the 💬 prompts below. Gemini can see your notebook. |
| **Generate in a cell** | The *generate* link in an empty code cell ("Start coding or generate with AI") | Quick, single-cell code requests |
| **Explain error** | The button under a cell that failed (sometimes next to **Fix error**) | Understanding and fixing an error |

**Every step follows the same loop:**
1. Copy the 💬 prompt into Gemini.
2. **Read** the code it suggests before you run it. Ask "what does this line do?" if you're unsure.
3. Put the code in the empty cell under the prompt (or let Gemini add it) and run it.
4. Compare your output to the ✅ checkpoint. Close is fine; very different means something is off.
5. If it's off, tell Gemini exactly what's wrong: *"The checkpoint says 1,002 rows but I got 1,008. Why?"*

If Gemini proposes a multi-step plan and offers to run it for you, read the plan before you approve it, then
check the results against the checkpoint like any other code.

> 🆘 **Behind or stuck?** Step 2 has a **catch-up cell**. Run it and you're back in sync. The later steps
> are optional for the evidence pack, so you can always finish the workshop.

## The Scenario

You're an analyst at an engine plant. The plant manager asks: **"Over the last two weeks, where are we losing
output and quality, and what should we fix first?"** They want a short, formatted report, not a notebook.

The plant runs three machines around the clock in three shifts. Each machine builds one engine model.
The data has one row per machine per hour.

| Column | Meaning |
|---|---|
| `Timestamp` | Start of the hour (YYYY-MM-DD HH:MM) |
| `Shift` | 1 = 07:00-14:59, 2 = 15:00-22:59, 3 = 23:00-06:59 (night) |
| `Day_of_Week` | Day name |
| `Machine_ID` | Machine: M1, M2, M3 |
| `Model_Type` | Engine model built: V6, V8, EV-1 |
| `Production_Count` | Engines produced in that hour |
| `Defective_Count` | Engines that failed inspection in that hour |
| `Downtime_Min` | Minutes the machine was stopped in that hour |

## Step 0: Load the Data (2 min, no AI needed)

Run the cell below. It loads the raw data as `df_raw` and shows the first rows.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_URL = ("https://raw.githubusercontent.com/NCSU-Libraries/ai_in_data_science/main/"
            "workshops/ise498/data/engine_production_data_original.csv")
LOCAL_FILE = Path("data/engine_production_data_original.csv")  # used if you cloned the repository

df_raw = pd.read_csv(LOCAL_FILE if LOCAL_FILE.exists() else DATA_URL)
print(f"{len(df_raw):,} rows x {df_raw.shape[1]} columns")
df_raw.head()

## Step 1: Trust Check (5 min)

Before analyzing anything, find out what's wrong with the data. Don't fix anything yet.

💬 **Prompt for Gemini**
```text
You are helping me analyze manufacturing data in this notebook. The DataFrame df_raw has one row per
machine per hour. Write one code cell that profiles data quality in df_raw and prints a short report:
1. number of rows and number of exact duplicate rows
2. missing values per column
3. the unique values of Shift, Day_of_Week, Machine_ID and Model_Type
4. any non-numeric values in Production_Count, Defective_Count and Downtime_Min
5. impossible values: negative counts, or more defects than engines produced
Do not change df_raw.
```

✅ **Checkpoint**: your output should reveal:
- 1,018 rows, 10 of them exact duplicates
- Missing values: 24 in `Defective_Count`, 21 in `Downtime_Min`
- Day names in two formats (`Monday` and `Mon`), and model `v6` next to `V6`
- An unknown machine, `M99` (3 rows)
- Text in `Downtime_Min`: `Maintenance` (5 rows)
- 6 impossible rows: 3 with negative production (-50) and 3 with 20 defects out of only 10 engines
  (a simple "defects > production" check also catches the negative rows, so it reports 6)

🤔 **Think:** which of these problems would change a plant manager's decision if you left it unfixed?

## Step 2: Clean the Data with Explicit Rules (7 min)

AI writes cleaning code quickly, but it can't know your plant's rules. **You decide the rules; Gemini writes
the code.** Notice how specific this prompt is.

💬 **Prompt for Gemini**
```text
Create a cleaned copy of df_raw named df_clean using exactly these rules, in this order:
1. Remove exact duplicate rows.
2. Expand abbreviated day names (Mon -> Monday, and so on) and make Model_Type uppercase.
3. Each machine builds one model (M1 = V6, M2 = V8, M3 = EV-1). Replace any Machine_ID that is not
   M1, M2 or M3 using its Model_Type.
4. Convert Downtime_Min to numbers. Text such as "Maintenance" becomes missing (NaN), not 0.
5. Convert Timestamp to datetime.
6. Drop rows with negative Production_Count or with Defective_Count greater than Production_Count.
7. Keep all other missing values as missing. Do not fill them with 0.
Also build a Python list of strings named cleaning_log, one sentence per rule, saying what changed and
how many rows it affected. Print len(df_clean) and the log.
```

✅ **Checkpoint**
- `df_clean` has **1,002 rows**, machines `M1`, `M2`, `M3` only, models `V6`, `V8`, `EV-1`, and 7 full day names
- `Downtime_Min` is numeric with 24 missing values, and `Defective_Count` has 24 missing values
- `cleaning_log` is a list of short sentences with row counts

💡 **Why rule 4 says NaN, not 0:** "Maintenance" means the machine *was* down; we just don't know for how
long. Filling with 0 would quietly erase downtime.

In [ ]:
# @title 🆘 Catch-up: Step 2 reference cleaning (run only if you're stuck or behind)
# Creates `df_clean` and `cleaning_log` exactly as the Step 2 prompt describes.
DAY_NAMES = {"Mon": "Monday", "Tue": "Tuesday", "Wed": "Wednesday", "Thu": "Thursday",
             "Fri": "Friday", "Sat": "Saturday", "Sun": "Sunday"}
MACHINE_FOR_MODEL = {"V6": "M1", "V8": "M2", "EV-1": "M3"}

df_clean = df_raw.copy()
cleaning_log = []

n = len(df_clean)
df_clean = df_clean.drop_duplicates()
cleaning_log.append(f"Removed {n - len(df_clean)} exact duplicate rows ({n:,} -> {len(df_clean):,} rows).")

short_days = df_clean["Day_of_Week"].isin(DAY_NAMES.keys()).sum()
lower_models = (df_clean["Model_Type"] != df_clean["Model_Type"].str.upper()).sum()
df_clean["Day_of_Week"] = df_clean["Day_of_Week"].replace(DAY_NAMES)
df_clean["Model_Type"] = df_clean["Model_Type"].str.upper()
cleaning_log.append(f"Standardized {short_days} abbreviated day names and {lower_models} lowercase model labels.")

unknown = ~df_clean["Machine_ID"].isin(["M1", "M2", "M3"])
df_clean.loc[unknown, "Machine_ID"] = df_clean.loc[unknown, "Model_Type"].map(MACHINE_FOR_MODEL)
cleaning_log.append(f"Reassigned {unknown.sum()} rows with unknown machine ID 'M99' using the "
                    "one-model-per-machine rule (V6 -> M1, V8 -> M2, EV-1 -> M3).")

text_downtime = df_clean["Downtime_Min"].notna() & pd.to_numeric(df_clean["Downtime_Min"], errors="coerce").isna()
df_clean["Downtime_Min"] = pd.to_numeric(df_clean["Downtime_Min"], errors="coerce")
cleaning_log.append(f"Converted Downtime_Min to numbers; {text_downtime.sum()} text entries ('Maintenance') "
                    "became missing because their duration is unknown.")

df_clean["Timestamp"] = pd.to_datetime(df_clean["Timestamp"])

n = len(df_clean)
impossible = (df_clean["Production_Count"] < 0) | (df_clean["Defective_Count"] > df_clean["Production_Count"])
df_clean = df_clean[~impossible].reset_index(drop=True)
cleaning_log.append(f"Dropped {n - len(df_clean)} impossible rows: negative production or more defects "
                    f"than engines produced ({n:,} -> {len(df_clean):,} rows).")

cleaning_log.append(f"Kept missing values as missing (not zero): {df_clean['Defective_Count'].isna().sum()} "
                    f"defect counts and {df_clean['Downtime_Min'].isna().sum()} downtime values.")

print(f"df_clean: {len(df_clean):,} rows")
for line in cleaning_log:
    print("-", line)

## Step 3: Baseline KPIs (5 min)

Establish the baseline before hunting for problems. Define each metric precisely, or the AI will choose a
definition for you.

💬 **Prompt for Gemini**
```text
Using df_clean, build a small KPI table (metric, value) with:
- total engines produced
- total defective engines
- defect rate = total defects / total production, using only rows where Defective_Count is not missing
- total downtime in minutes (ignore missing values) and the same in hours
- share of machine time lost to downtime = total downtime minutes / (60 x number of rows with known downtime)
Format percentages with 2 decimals.
```

✅ **Checkpoint**
- 36,888 engines produced, 439 defective
- Defect rate **1.22%** (if you got 1.19%, your denominator includes rows whose defect count is unknown; ask Gemini to fix it)
- Downtime 3,333 minutes (55.55 hours), or **5.7%** of machine time

## Step 4: Where Does the Loss Concentrate? (8 min)

A plant-wide average hides where problems live. Break the KPIs down by machine and by shift, then follow
the strongest signal.

💬 **Prompt 4a for Gemini**
```text
Using df_clean, make two summary tables, one by Machine_ID and one by Shift, each with: engines produced,
defects, defect rate (rows with known defects only), share of all defects, total downtime minutes and
share of all downtime. Then plot defect rate by shift and downtime minutes by machine as two bar charts.
```

✅ **Checkpoint 4a**
- **Shift 3** (night, 23:00-06:59): defect rate about **2.4%**, versus about 0.6% on shifts 1 and 2. It makes about two-thirds of all defects.
- **M2** (builds the V8): about **62%** of all downtime minutes, and the highest defect rate (about 2.1%)

💬 **Prompt 4b for Gemini** (drill down)
```text
Two follow-ups on df_clean:
1. Compute the defect rate for each hour of the day (0-23) and plot it as a bar chart. Which hours stand out?
2. Downtime_Min values look like either short stops (5-10 min) or long stoppages (20+ min). Count the long
   stoppages (Downtime_Min >= 20) by machine and show their total minutes.
```

✅ **Checkpoint 4b**
- The defect rate jumps to **3.4-4.4% from 03:00 to 06:59**, the end of the night shift. Every other hour is under 1%.
- All **44 long stoppages** (20-58 min each) happen on **M2**: 1,558 minutes, nearly half of all downtime.

🤔 **Think:** is this one problem or two? What would you go and look at on the plant floor?

### ⏱️ Stretch (only if you have time): Is the Night-Shift Gap Real?

💬 **Prompt for Gemini**
```text
Is the defect rate really different between shifts, or could it be chance? Compute each row's defect rate
(Defective_Count / Production_Count, rows with known defects only), run a Kruskal-Wallis test across the
three shifts, and explain the result in plain language.
```

✅ **Checkpoint:** H ≈ 98 and p < 0.001. The evidence pack runs this test for you too.

## Step 5: Write Your Takeaways (4 min)

Gemini computed the numbers. **Interpreting them is your job.** Write 3-4 takeaways in your own words, each
pointing to a number. They go into the evidence pack, and Claude builds the report around *your* reading
of the data (and flags anything the numbers don't support).

Tip: ask Gemini to critique them first: *"Here are my takeaways: ... Which claims are not supported by the
outputs in this notebook?"*

In [ ]:
my_takeaways = [
    "Replace this with takeaway 1 ...",
    "Replace this with takeaway 2 ...",
    "Replace this with takeaway 3 ...",
]

## Step 6: Build the Evidence Pack (3 min)

Run the cell below. It checks `df_clean`, computes every number the report needs, draws three
report-ready charts and downloads **`evidence_pack.zip`**:

```text
evidence_pack/
├── findings.md        all numbers + your cleaning log + your takeaways
└── charts/
    ├── fig1_defect_rate_by_hour.png
    ├── fig2_downtime_by_machine.png
    └── fig3_defect_rate_machine_shift.png
```

**Why a separate pack?** It's a contract. The report writer gets verified numbers, not your scratch work.
If a number isn't in `findings.md`, it doesn't belong in the report.

> If the download doesn't start, open the 📁 **Files** panel on the left, right-click `evidence_pack.zip`
> and choose **Download**.

In [ ]:
# @title 📦 Build and download the evidence pack (just run this cell)
# Turns `df_clean` (+ `cleaning_log` and `my_takeaways`) into:
#   evidence_pack/findings.md    every number the report may use, computed by code
#   evidence_pack/charts/*.png   three report-ready charts
# and downloads evidence_pack.zip. Nothing to edit here; expand the cell to read the code.
import shutil
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import PercentFormatter
from scipy import stats

PACK_DIR = Path("evidence_pack")
LONG_STOP_MIN = 20  # downtime is either a routine stop (5-10 min) or a long stoppage (20+ min)

# Chart colors: one accent for what matters, gray for context, a blue ramp for magnitude
SURFACE, INK, INK_SOFT, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
ACCENT, CONTEXT = "#2a78d6", "#c3c2b7"
BLUES = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]


def pct(x, digits=1):
    return "n/a" if pd.isna(x) else f"{x:.{digits}%}"


def num(x, digits=0):
    return "n/a" if pd.isna(x) else f"{x:,.{digits}f}"


def md_table(header, rows, left=1):
    """Markdown table: the first `left` columns are left-aligned, the rest right-aligned (numbers)."""
    lines = ["| " + " | ".join(header) + " |",
             "|" + "---|" * left + "---:|" * (len(header) - left)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in rows]
    return "\n".join(lines)


def hour_blocks(hours):
    """Split clock hours into consecutive blocks, wrapping past midnight."""
    hours, blocks = {int(h) for h in hours}, []
    while hours:
        start = min(hours)
        while (start - 1) % 24 in hours and len(hours) < 24:
            start = (start - 1) % 24
        block = []
        while start in hours:
            block.append(start)
            hours.remove(start)
            start = (start + 1) % 24
        blocks.append(block)
    return blocks


def hour_label(block):
    return f"{block[0]:02d}:00–{block[-1]:02d}:59"


def check_clean_data(df):
    needed = ["Timestamp", "Shift", "Machine_ID", "Model_Type",
              "Production_Count", "Defective_Count", "Downtime_Min"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"df_clean has no column(s) {missing}. "
                         "Run the Step 2 catch-up cell, then run this cell again.")
    d = df[needed].copy()
    d["Timestamp"] = pd.to_datetime(d["Timestamp"], errors="coerce")
    for col in ["Shift", "Production_Count", "Defective_Count", "Downtime_Min"]:
        d[col] = pd.to_numeric(d[col], errors="coerce")
    checks = {
        "No duplicate rows": not d.duplicated().any(),
        "One row per machine per hour": not d.duplicated(["Timestamp", "Machine_ID"]).any(),
        "Machines are M1, M2, M3 only": set(d["Machine_ID"]) <= {"M1", "M2", "M3"},
        "Every timestamp is a valid date": d["Timestamp"].notna().all(),
        "Production counts present and never negative": (d["Production_Count"] >= 0).all(),
        "Defects never exceed production": not (d["Defective_Count"] > d["Production_Count"]).any(),
    }
    for name, ok in checks.items():
        print(("✅ " if ok else "❌ ") + name)
    if not all(checks.values()):
        raise ValueError("df_clean failed a check above. Fix your Step 2 cleaning "
                         "(or run the Step 2 catch-up cell), then run this cell again.")
    d["Shift"] = d["Shift"].astype(int)
    return d


def group_stats(d, by):
    has_def = d["Defective_Count"].notna()
    g = d.groupby(by)
    out = pd.DataFrame({
        "machine_hours": g.size(),
        "engines": g["Production_Count"].sum(),
        "engines_per_hour": g["Production_Count"].mean(),
        "defects": g["Defective_Count"].sum(),
        "defect_base": d[has_def].groupby(by)["Production_Count"].sum(),
        "downtime": g["Downtime_Min"].sum(),
        "downtime_per_hour": g["Downtime_Min"].mean(),
        "long_stops": g["Long_Stop"].sum(),
    })
    out["defect_rate"] = out["defects"] / out["defect_base"]
    out["defect_share"] = out["defects"] / out["defects"].sum()
    out["downtime_share"] = out["downtime"] / out["downtime"].sum()
    return out


def style_axes(ax, grid_axis):
    ax.set_facecolor(SURFACE)
    for side in ["top", "right", "left", "bottom"]:
        ax.spines[side].set_visible(False)
    ax.tick_params(length=0, labelcolor=INK_SOFT, labelsize=9)
    if grid_axis:
        ax.grid(axis=grid_axis, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)


def add_titles(fig, title, subtitle):
    fig.text(0.01, 0.99, title, ha="left", va="top", fontsize=12, fontweight="bold", color=INK)
    fig.text(0.01, 0.91, subtitle, ha="left", va="top", fontsize=9, color=INK_SOFT)


def save(fig, path):
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()
    plt.close(fig)


def build_evidence_pack(df, cleaning_log=None, takeaways=None, out_dir=PACK_DIR):
    d = check_clean_data(df)
    has_def = d["Defective_Count"].notna()
    d["Hour"] = d["Timestamp"].dt.hour
    d["Long_Stop"] = d["Downtime_Min"] >= LONG_STOP_MIN

    # Shifts, and the plant day (starts when the first shift starts)
    shift_blocks = {s: hour_blocks(h)[0] for s, h in d.groupby("Shift")["Hour"].unique().items()}
    day_start = shift_blocks[min(shift_blocks)][0]
    d["Plant_Day"] = (d["Timestamp"] - pd.Timedelta(hours=day_start)).dt.date
    n_days = d["Plant_Day"].nunique()
    shift_name = {s: f"Shift {s} ({hour_label(b)})" for s, b in shift_blocks.items()}
    model_of = d.groupby("Machine_ID")["Model_Type"].agg(lambda s: ", ".join(sorted(s.unique())))

    # Headline numbers
    engines, defects = d["Production_Count"].sum(), d["Defective_Count"].sum()
    defect_rate = defects / d.loc[has_def, "Production_Count"].sum()
    downtime = d["Downtime_Min"].sum()
    time_lost = downtime / (d["Downtime_Min"].notna().sum() * 60)
    by_machine = group_stats(d, "Machine_ID")
    by_shift = group_stats(d, "Shift")
    by_machine_shift = group_stats(d, ["Machine_ID", "Shift"])["defect_rate"].unstack()
    by_hour = group_stats(d, "Hour")
    by_hour["shift"] = d.groupby("Hour")["Shift"].agg(lambda s: s.mode().iat[0])

    # When defects happen: hours above 2x the median hourly defect rate
    median_rate = by_hour["defect_rate"].median()
    elevated = by_hour.index[by_hour["defect_rate"] > 2 * median_rate]
    blocks = hour_blocks(elevated)
    peak = max(blocks, key=lambda b: by_hour.loc[b, "defects"].sum()) if blocks else []
    in_peak = d["Hour"].isin(peak)

    def rate(mask):
        rows = d[mask & has_def]
        return rows["Defective_Count"].sum() / rows["Production_Count"].sum()

    peak_rate, rest_rate = (rate(in_peak), rate(~in_peak)) if peak else (float("nan"),) * 2
    peak_defects = d.loc[in_peak, "Defective_Count"].sum()
    excess = peak_defects - d.loc[in_peak & has_def, "Production_Count"].sum() * rest_rate

    daily = d[has_def].groupby(["Plant_Day", "Shift"])[["Defective_Count", "Production_Count"]].sum()
    worst_shift_days = (daily["Defective_Count"] / daily["Production_Count"]).unstack().idxmax(axis=1).value_counts()

    # Long stoppages and the output they cost
    stops = d[d["Long_Stop"]]
    routine = d[(d["Downtime_Min"] > 0) & ~d["Long_Stop"]]
    stops_per_day = stops.groupby("Plant_Day").size()
    lost = {}
    for machine, rows in d.groupby("Machine_ID"):
        if rows["Long_Stop"].any():
            normal = rows.loc[~rows["Long_Stop"], "Production_Count"].mean()
            during = rows.loc[rows["Long_Stop"], "Production_Count"]
            lost[machine] = (normal, during.mean(), (normal - during).sum())

    # Statistical check: do per machine-hour defect rates differ by shift?
    rated = d[has_def & (d["Production_Count"] > 0)]
    hourly_rate = rated["Defective_Count"] / rated["Production_Count"]
    shifts = sorted(rated["Shift"].unique())
    h_stat, p_value = stats.kruskal(*[hourly_rate[rated["Shift"] == s] for s in shifts])

    one_to_one = (d.groupby("Machine_ID")["Model_Type"].nunique().eq(1).all()
                  and d.groupby("Model_Type")["Machine_ID"].nunique().eq(1).all())

    # ---- Charts ------------------------------------------------------------
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    (out_dir / "charts").mkdir(parents=True)
    charts = []

    # Figure 1: defect rate by hour of day, plant-day order, peak hours highlighted
    order = [(day_start + i) % 24 for i in range(24)]
    hourly = by_hour.reindex(order)
    fig, ax = plt.subplots(figsize=(8, 3.6), facecolor=SURFACE)
    fig.subplots_adjust(top=0.80)
    ax.bar(range(24), hourly["defect_rate"], width=0.62,
           color=[ACCENT if h in peak else CONTEXT for h in order])
    style_axes(ax, "y")
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_xticks(range(24), [f"{h:02d}" for h in order])
    ax.set_xlim(-0.6, 23.6)
    for s, block in shift_blocks.items():
        idx = [order.index(h) for h in block]
        ax.text((min(idx) + max(idx)) / 2, -0.17, shift_name[s], transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=9, color=INK_SOFT)
        if min(idx) > 0:
            ax.axvline(min(idx) - 0.5, color=AXIS, linewidth=0.8)
    if peak:
        last = max(order.index(h) for h in peak)
        ax.annotate(f"{hour_label(peak)}: {pct(peak_rate)} defect rate\n"
                    f"{peak_rate / rest_rate:.1f}× the rest of the day ({pct(rest_rate)})",
                    xy=(last + 0.4, hourly["defect_rate"].max()), ha="right", va="bottom",
                    fontsize=9, color=INK, xytext=(0, 6), textcoords="offset points")
        ax.set_ylim(0, hourly["defect_rate"].max() * 1.32)
    add_titles(fig, "Defect rate by hour of day",
               "Hours run in plant-day order. Highlighted: hours above twice the median hourly rate.")
    path = out_dir / "charts" / "fig1_defect_rate_by_hour.png"
    save(fig, path)
    charts.append((path, "Defect rate for each hour of the day, grouped by shift; elevated hours highlighted",
                   "Section 7"))

    # Figure 2: downtime minutes by machine, routine stops vs long stoppages
    machines = list(by_machine.index)[::-1]
    long_min = stops.groupby("Machine_ID")["Downtime_Min"].sum().reindex(machines, fill_value=0)
    routine_min = by_machine["downtime"].reindex(machines) - long_min
    fig, ax = plt.subplots(figsize=(8, 2.7), facecolor=SURFACE)
    fig.subplots_adjust(top=0.70)
    ys = range(len(machines))
    ax.barh(ys, routine_min, height=0.5, color=CONTEXT, edgecolor=SURFACE, linewidth=1.5,
            label=f"Routine stops ({num(routine['Downtime_Min'].min())}–{num(routine['Downtime_Min'].max())} min each)")
    ax.barh(ys, long_min, left=routine_min, height=0.5, color=ACCENT, edgecolor=SURFACE, linewidth=1.5,
            label=f"Long stoppages (≥ {LONG_STOP_MIN} min each)")
    style_axes(ax, "x")
    ax.set_yticks(list(ys), [f"{m} ({model_of[m]})" for m in machines])
    ax.xaxis.set_major_formatter(lambda x, _: f"{x:,.0f}")
    ax.set_xlabel("Downtime (minutes)", fontsize=9, color=INK_SOFT)
    for y, m in zip(ys, machines):
        note = f"{num(by_machine.loc[m, 'downtime'])} min"
        if by_machine.loc[m, "long_stops"]:
            note += f" · {int(by_machine.loc[m, 'long_stops'])} long stoppages"
        ax.text(by_machine.loc[m, "downtime"], y, "  " + note, va="center", fontsize=9, color=INK)
    ax.set_xlim(0, by_machine["downtime"].max() * 1.45)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, fontsize=9,
              handlelength=1.0, handleheight=1.0, labelcolor=INK_SOFT)
    add_titles(fig, "Downtime by machine", "Total recorded downtime over the period, split by stop length.")
    path = out_dir / "charts" / "fig2_downtime_by_machine.png"
    save(fig, path)
    charts.append((path, "Downtime minutes per machine, routine stops vs long stoppages", "Sections 4 and 8"))

    # Figure 3: defect rate heatmap, machine x shift
    grid = by_machine_shift.reindex(index=list(by_machine.index), columns=sorted(shift_blocks))
    fig, ax = plt.subplots(figsize=(7, 2.9), facecolor=SURFACE)
    fig.subplots_adjust(top=0.78)
    vmax = grid.max().max()
    ax.imshow(grid.values, cmap=LinearSegmentedColormap.from_list("blues", BLUES),
              vmin=0, vmax=vmax, aspect="auto")
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            v = grid.iat[i, j]
            ax.text(j, i, pct(v), ha="center", va="center", fontsize=10,
                    color="white" if v > 0.5 * vmax else INK)
    ax.set_xticks(range(grid.shape[1]), [shift_name[s].replace(" (", "\n(") for s in grid.columns])
    ax.set_yticks(range(grid.shape[0]), [f"{m} ({model_of[m]})" for m in grid.index])
    ax.set_xticks([x - 0.5 for x in range(1, grid.shape[1])], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, grid.shape[0])], minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=3)
    ax.tick_params(which="both", length=0, labelcolor=INK_SOFT, labelsize=9)
    for side in ax.spines.values():
        side.set_visible(False)
    add_titles(fig, "Defect rate by machine and shift", "Darker = higher defect rate.")
    path = out_dir / "charts" / "fig3_defect_rate_machine_shift.png"
    save(fig, path)
    charts.append((path, "Defect rate for every machine and shift combination", "Section 6"))

    # ---- findings.md -------------------------------------------------------
    start, end = d["Timestamp"].min(), d["Timestamp"].max()
    md = [f"# Evidence pack: engine production, {start:%b} {start.day} to {end:%b} {end.day}, {end:%Y}", "",
          f"Every number in this file was computed by code from the cleaned dataset on {datetime.now():%Y-%m-%d %H:%M}. "
          "It is the single source of truth for the report.", ""]

    md += ["## 1. Dataset", "", md_table(["Item", "Value"], [
        ["Source file", "engine_production_data_original.csv"],
        ["Period", f"{start:%Y-%m-%d %H:%M} to {end:%Y-%m-%d %H:%M} ({n_days} plant days; a plant day starts at {day_start:02d}:00)"],
        ["Unit of observation", "one machine-hour (one row per machine per hour)"],
        ["Rows after cleaning", f"{len(d):,} machine-hours"],
        ["Machines (model built)", ", ".join(f"{m} ({model_of[m]})" for m in by_machine.index)],
        ["Shifts", "; ".join(shift_name[s] for s in sorted(shift_name))],
    ], left=2), ""]

    md += ["## 2. Data cleaning log", ""]
    md += [f"- {line}" for line in (cleaning_log or ["(No cleaning log was provided.)"])]
    md += ["", "How missing values are handled: defect rates use only rows with a known defect count, "
               "and downtime totals use only rows with a known downtime value.", ""]

    md += ["## 3. Headline KPIs", "", md_table(["Metric", "Value"], [
        ["Engines produced", num(engines)],
        ["Average engines per machine-hour", num(d["Production_Count"].mean(), 1)],
        ["Defective engines", num(defects)],
        ["Defect rate", pct(defect_rate, 2)],
        ["Downtime recorded", f"{num(downtime)} min ({num(downtime / 60, 2)} h)"],
        [f"Share of machine time lost to downtime (of {num(d['Downtime_Min'].notna().sum())} machine-hours "
         "with a known downtime value)", pct(time_lost)],
        [f"Long stoppages (≥ {LONG_STOP_MIN} min)", num(len(stops))],
    ]), ""]

    md += ["## 4. By machine", "", md_table(
        ["Machine", "Model", "Machine-hours", "Engines", "Engines per hour", "Defects", "Defect rate",
         "Share of defects", "Downtime (min)", "Share of downtime", "Long stoppages"],
        [[m, model_of[m], num(r.machine_hours), num(r.engines), num(r.engines_per_hour, 1), num(r.defects),
          pct(r.defect_rate, 2), pct(r.defect_share), num(r.downtime), pct(r.downtime_share), num(r.long_stops)]
         for m, r in by_machine.iterrows()], left=2), ""]

    md += ["## 5. By shift", "", md_table(
        ["Shift", "Hours", "Machine-hours", "Engines", "Defects", "Defect rate", "Share of defects",
         "Downtime (min)", "Avg downtime per machine-hour (min)", "Long stoppages"],
        [[f"Shift {s}", hour_label(shift_blocks[s]), num(r.machine_hours), num(r.engines), num(r.defects),
          pct(r.defect_rate, 2), pct(r.defect_share), num(r.downtime), num(r.downtime_per_hour, 1),
          num(r.long_stops)] for s, r in by_shift.iterrows()], left=2), ""]

    md += ["## 6. Defect rate by machine and shift", "", "Table view of Figure 3.", "", md_table(
        ["Machine"] + [f"Shift {s}" for s in grid.columns],
        [[f"{m} ({model_of[m]})"] + [pct(v, 2) for v in grid.loc[m]] for m in grid.index]), ""]

    md += ["## 7. When defects happen: hour of day", ""]
    if peak:
        top_shift, top_days = worst_shift_days.index[0], worst_shift_days.iloc[0]
        md += [f"- Median hourly defect rate: {pct(median_rate, 2)}. Elevated hours (more than twice the median): "
               f"{', '.join(hour_label(b) for b in blocks)}.",
               f"- The main elevated window, {hour_label(peak)}, lies in Shift {by_hour.loc[peak[0], 'shift']} and covers "
               f"{len(peak)} of 24 hours ({pct(len(peak) / 24)} of the day).",
               f"- Defects in that window: {num(peak_defects)} of {num(defects)} ({pct(peak_defects / defects)}).",
               f"- Defect rate in that window: {pct(peak_rate, 2)}, versus {pct(rest_rate, 2)} in all other hours "
               f"({peak_rate / rest_rate:.1f}×).",
               f"- Estimated excess defects in that window (defects above what the rest-of-day rate predicts): "
               f"about {num(excess)}.",
               "- Defect rate in that window by machine: " + "; ".join(
                   f"{m} {pct(rate(in_peak & (d['Machine_ID'] == m)), 2)} vs {pct(rate(~in_peak & (d['Machine_ID'] == m)), 2)} "
                   "in other hours" for m in by_machine.index) + ".",
               f"- Shift {top_shift} had the highest defect rate of the {len(shifts)} shifts on {top_days} of {n_days} plant days.",
               ""]
    md += ["Table view of Figure 1. Defect rates use only rows with a known defect count, so they can "
           "differ slightly from defects ÷ engines.", "", md_table(
        ["Hour", "Shift", "Engines", "Defects", "Defect rate"],
        [[f"{h:02d}:00", f"Shift {by_hour.loc[h, 'shift']}", num(by_hour.loc[h, "engines"]),
          num(by_hour.loc[h, "defects"]), pct(by_hour.loc[h, "defect_rate"], 2)] for h in order], left=2), ""]

    md += [f"## 8. Long stoppages (≥ {LONG_STOP_MIN} min)", ""]
    if len(stops):
        md += [f"- {len(stops)} long stoppages, {num(stops['Downtime_Min'].sum())} min in total "
               f"({pct(stops['Downtime_Min'].sum() / downtime)} of all recorded downtime); each lasted "
               f"{num(stops['Downtime_Min'].min())}–{num(stops['Downtime_Min'].max())} min.",
               "- Long stoppages by machine: " + ", ".join(
                   f"{m}: {num(by_machine.loc[m, 'long_stops'])}" for m in by_machine.index) + ".",
               f"- They happened on {stops['Plant_Day'].nunique()} of {n_days} plant days "
               f"({stops_per_day.min()}–{stops_per_day.max()} per day on days with a stoppage)."]
        for m, (normal, during, units) in lost.items():
            md += [f"- {m} output: {num(during, 1)} engines per hour during long-stoppage hours vs "
                   f"{num(normal, 1)} in its other hours. Estimated engines lost: about {num(units)} "
                   f"({pct(units / by_machine.loc[m, 'engines'])} of {m}'s actual output)."]
    else:
        md += ["- No long stoppages in the data."]
    md += [f"- Routine stops (under {LONG_STOP_MIN} min): {len(routine)} stops, {num(routine['Downtime_Min'].sum())} min "
           f"in total, {num(routine['Downtime_Min'].min())}–{num(routine['Downtime_Min'].max())} min each, on "
           f"{routine['Machine_ID'].nunique()} machines.", ""]

    md += ["## 9. Statistical check", "",
           "- Question: do defect rates differ between shifts?",
           "- Test: Kruskal-Wallis H test (non-parametric) on the defect rate of each machine-hour, grouped by shift.",
           "- Sample sizes (machine-hours with a known defect count): "
           + ", ".join(f"Shift {s}: {num((rated['Shift'] == s).sum())}" for s in shifts) + ".",
           f"- Result: H = {h_stat:.1f}, p {'< 0.001' if p_value < 0.001 else f'= {p_value:.3f}'} (significance level 0.05).",
           ""]

    md += ["## 10. What this data cannot tell us", ""]
    if one_to_one:
        md += ["- Each machine builds exactly one model (" + ", ".join(f"{m} = {model_of[m]}" for m in by_machine.index)
               + "), so machine effects and model effects cannot be separated."]
    md += [f"- The period is {n_days} plant days; longer-term or seasonal patterns are not visible.",
           "- There are no downtime reason codes, defect types, staffing, or maintenance records, so causes "
           "can only be hypothesized.",
           f"- {d['Defective_Count'].isna().sum()} defect counts and {d['Downtime_Min'].isna().sum()} "
           "downtime values are missing.", ""]

    written = [t for t in (takeaways or []) if t and not t.startswith("Replace this")]
    md += ["## 11. Analyst takeaways (written by the analyst, not computed)", ""]
    md += [f"- {t}" for t in written] or ["- (No takeaways were provided.)"]
    md += ["", "## 12. Charts", "", md_table(
        ["File", "What it shows", "Table view"],
        [[f"`charts/{p.name}`", what, table] for p, what, table in charts], left=3), ""]

    (out_dir / "findings.md").write_text("\n".join(md), encoding="utf-8")
    zip_path = shutil.make_archive(str(out_dir), "zip", root_dir=out_dir)
    print(f"\n📦 Evidence pack ready: {Path(zip_path).name} (findings.md + {len(charts)} charts)")
    try:
        from google.colab import files
        files.download(zip_path)
    except ImportError:
        print(f"Saved next to this notebook: {Path(zip_path).name} and the {out_dir}/ folder.")
    return out_dir


if "df_clean" not in globals():
    raise NameError("There is no df_clean yet. Finish Step 2 (or run its catch-up cell) first.")
pack = build_evidence_pack(df_clean, globals().get("cleaning_log"), globals().get("my_takeaways"))

## ✅ Part 1 Done. Next: Write the Report with Claude

1. Unzip `evidence_pack.zip` on your computer.
2. Open the Part 2 guide: **[Write the report with Claude](https://github.com/NCSU-Libraries/ai_in_data_science/blob/main/workshops/ise498/report_with_claude.md)**
3. Keep this notebook open. You'll use it to fact-check Claude's report.